# Segmentação de Audiências B2B com K-Means

Pipeline completo para identificar perfis comportamentais de contas B2B em mídia paga.
O modelo separa automaticamente as empresas em 3 clusters com estratégias de mídia distintas,
eliminando o desperdício de budget que ocorre quando todas as contas recebem o mesmo tratamento.

**Variáveis utilizadas:** CTR, CPC médio, Tempo médio de sessão no site

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_b2b_engagement_data
from src.preprocessing import aggregate_by_company, remove_outliers, scale_features
from src.clustering import run_elbow_method, fit_kmeans, calculate_silhouette
from src.visualization import plot_correlation_matrix, plot_elbow, plot_clusters_scatter
from src.inference import save_model, load_model, classify_and_act

pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Geração dos dados

Simulamos 8.000 eventos de interação de 1.000 empresas B2B com anúncios.
Cada linha representa uma impressão com informações de clique, custo e duração de sessão.

In [ ]:
raw_df = generate_b2b_engagement_data(n_events=8000, n_companies=1000, random_state=42)
print(f"Eventos gerados: {len(raw_df):,}")
print(f"Empresas únicas: {raw_df['company_id'].nunique():,}")
raw_df.head()

## 2. Agregação por empresa

Calculamos as métricas de negócio no nível da conta corporativa:
- **CTR:** taxa de clique — captura intenção de interação
- **CPC:** custo por clique médio — reflete competitividade no leilão
- **avg_time_site:** tempo médio de sessão em minutos — proxy de interesse real

In [ ]:
df_company = aggregate_by_company(raw_df)
print(f"Empresas após agregação: {len(df_company):,}")
df_company.head(10)

## 3. Análise Exploratória

Verificamos a distribuição das variáveis antes de qualquer transformação.

In [ ]:
features = ['ctr', 'cpc', 'avg_time_site']
df_company[features].describe()

## 4. Matriz de Correlação

Validamos que as três variáveis capturam dimensões comportamentais independentes.
Correlação alta entre variáveis tornaria a clusterização redundante.

In [ ]:
plot_correlation_matrix(df_company, features)

## 5. Remoção de Outliers e Normalização

K-Means é sensível a valores extremos. Removemos acima do percentil 99 e padronizamos as escalas.

In [ ]:
df_clean = remove_outliers(df_company, features, percentile=99)
print(f"Empresas após remoção de outliers: {len(df_clean):,}")

X_scaled, scaler = scale_features(df_clean, features)
print(f"Shape do dataset normalizado: {X_scaled.shape}")

## 6. Método do Cotovelo — definindo o k ideal

Testamos de k=1 a k=10 e identificamos o ponto onde a redução de inércia se torna marginal.

In [ ]:
wcss = run_elbow_method(X_scaled)
plot_elbow(wcss)

## 7. Clustering com k=3

A queda de inércia é acentuada até k=3. A partir de 4 clusters, o ganho é marginal.
Executamos o modelo definitivo e validamos com Silhouette Score.

In [ ]:
kmeans = fit_kmeans(X_scaled, n_clusters=3)
df_clean = df_clean.copy()
df_clean['cluster'] = kmeans.labels_

score = calculate_silhouette(X_scaled, kmeans.labels_)
print(f"Silhouette Score: {score:.2f}")
print(f"Distribuição por cluster:")
print(df_clean['cluster'].value_counts().sort_index())

## 8. Visualização dos Clusters

Dispersão das contas B2B no espaço CTR × Tempo de Sessão, coloridas por cluster.

In [ ]:
plot_clusters_scatter(df_clean)

## 9. Perfis por Cluster

Médias das métricas de engajamento por cluster para construir as personas de audiência.

In [ ]:
profile = df_clean.groupby('cluster')[features].mean().round(2)
profile.index.name = 'Cluster'
profile.columns = ['CTR (%)', 'CPC (R$)', 'Tempo Médio (min)']

cluster_names = {
    0: 'Alta Performance',
    1: 'Alto Custo Estratégico',
    2: 'Volume Sem Retorno'
}
profile['Perfil'] = profile.index.map(cluster_names)
profile

## 10. Plano de Ação por Cluster

### Cluster 0 — Alta Performance
Alto CTR, CPC moderado. Esse é o cluster que justifica o investimento em mídia paga B2B.
- Maximizar budget nesse segmento
- Criar Lookalike Audiences com as empresas desse cluster nas plataformas
- Usar lances automatizados de CPA ou Maximizar Conversões

### Cluster 1 — Alto Custo Estratégico
CPC inflado (76% mais caro que o Cluster 0), tempo de sessão alto. A empresa tem interesse mas não converte direto.
- Remover das campanhas de fundo de funil
- Capturar via retargeting com material de alto valor (e-book, webinar, ferramenta gratuita)
- Transferir para automação de e-mail: custo marginal próximo de zero

### Cluster 2 — Volume Sem Retorno
CTR praticamente zero, menor tempo de sessão. Consome budget sem retorno mensurável.
- Auditar termos de busca que atraem esse perfil
- Negativar palavras-chave que geram esse tráfego
- Se CTR não melhorar após A/B test de 14 dias: excluir e mover verba para o Cluster 0

## 11. Deployment — Serialização e Inferência em Produção

In [ ]:
save_model(kmeans, scaler)
print("Modelo salvo em models/modelo_b2b.pkl")
print("Scaler salvo em models/scaler_b2b.pkl")

In [ ]:
# Simula uma nova empresa chegando via CRM
nova_empresa = {
    'ctr': 15.2,
    'cpc': 41.0,
    'avg_time_site': 2.1
}

modelo, padronizador = load_model()
resultado = classify_and_act(nova_empresa, modelo, padronizador, features)

print(f"Cluster: {resultado['cluster']}")
print(f"Ação recomendada: {resultado['action']}")